# Scene Detection Notebook
Test and tune the cosine-similarity scene detection pipeline.

**Cell 1** — cosine similarity function (edit threshold here)  
**Cell 2** — cut at keyframes, generate thumbnails, run pair comparison

In [8]:
import numpy as np
from PIL import Image
import os

# ── CONFIG ────────────────────────────────────────────────────────────────────
DISSIM_THRESHOLD = 0.10  # dissimilarity threshold for a cut (tune this)
POOL_DIM         = 8     # mosaic block size; larger = coarser, more robust

# ── Helpers ───────────────────────────────────────────────────────────────────
def pooling(arr: np.ndarray, dim: int) -> np.ndarray:
    """Average-pool (H, W, C) or (H, W) by block size `dim`."""
    h = (arr.shape[0] // dim) * dim
    w = (arr.shape[1] // dim) * dim
    arr = arr[:h, :w]
    if arr.ndim == 3:
        c = arr.shape[2]
        return arr.reshape(h // dim, dim, w // dim, dim, c).mean(axis=(1, 3))
    return arr.reshape(h // dim, dim, w // dim, dim).mean(axis=(1, 3))

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    a_f = a.flatten().astype(np.float32)
    b_f = b.flatten().astype(np.float32)
    denom = np.linalg.norm(a_f) * np.linalg.norm(b_f)
    if denom == 0:
        return 1.0
    return float(np.dot(a_f, b_f) / denom)

def check_pair_similar(path_a: str, path_b: str, threshold: float = DISSIM_THRESHOLD) -> bool:
    try:
        img_a = np.array(Image.open(path_a).convert("RGB"))
        img_b = np.array(Image.open(path_b).convert("RGB"))
    except Exception:
        return False

    sim      = cosine_similarity(pooling(img_a, POOL_DIM), pooling(img_b, POOL_DIM))
    dissim   = 1.0 - sim

    print(
        f"{os.path.basename(path_a)} ↔ {os.path.basename(path_b)} | "
        f"sim={sim:.4f}  dissim={dissim:.4f}  "
        f"→ {'MERGE' if dissim < threshold else 'CUT'}"
    )
    return dissim < threshold

In [ ]:
import os
import subprocess
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

import av
from PIL import Image as PILImage

# ── CONFIG ────────────────────────────────────────────────────────────────────
VIDEO_PATH         = r"../test/CSM_buggy_movie.mkv"  # <-- change this
OUTPUT_DIR         = "../test_out"
THUMB_WIDTH        = 360
THUMB_QUALITY      = 80
MIN_SCENE_DURATION = 2.0
MAX_THUMB_WORKERS  = 4

os.makedirs(OUTPUT_DIR, exist_ok=True)
FILE_NAME = os.path.splitext(os.path.basename(VIDEO_PATH))[0]

# ── Helpers ───────────────────────────────────────────────────────────────────
def fmt_ts(seconds: float) -> str:
    v = f"{float(seconds):.6f}"
    return v.rstrip("0").rstrip(".")

def get_duration(path: str) -> float:
    with av.open(path) as c:
        if c.duration is not None:
            return float(c.duration / av.time_base)
        stream = c.streams.video[0]
        if stream.duration is not None and stream.time_base is not None:
            return float(stream.duration * stream.time_base)
        raise RuntimeError(f"Could not determine duration for {path}")

def merge_short(boundaries: list, min_dur: float) -> list:
    if len(boundaries) <= 2:
        return boundaries
    merged = [boundaries[0]]
    for ts in boundaries[1:]:
        if ts - merged[-1] >= min_dur:
            merged.append(ts)
    return merged

# ── Step 1: Extract keyframes ─────────────────────────────────────────────────
print("PROGRESS|10|Extracting keyframes...")
keyframe_times = []
with av.open(VIDEO_PATH) as container:
    stream = container.streams.video[0]
    for pkt in container.demux(stream):
        if pkt.pts is None:
            continue
        if stream.time_base is not None:
            keyframe_times.append(float(pkt.pts * stream.time_base))

keyframe_times = sorted(set(keyframe_times))
print(f"  {len(keyframe_times)} keyframes found")

# ── Step 2: Cut at keyframes ──────────────────────────────────────────────────
cut_points = sorted(keyframe_times[1:])
cut_points = merge_short([0.0] + cut_points, MIN_SCENE_DURATION)[1:]
print(f"PROGRESS|50|Cutting {len(cut_points)} scenes...")

output_pattern = os.path.join(OUTPUT_DIR, f"{FILE_NAME}_%04d.mp4")
ffmpeg_cmd = [
    "ffmpeg", "-y", "-i", VIDEO_PATH,
    "-c", "copy",
    "-f", "segment",
    "-segment_times", ",".join(fmt_ts(p) for p in cut_points),
    "-reset_timestamps", "1",
    output_pattern,
]
result = subprocess.run(ffmpeg_cmd, capture_output=True, text=True)
if result.returncode != 0:
    print("ffmpeg error:", result.stderr[-1000:])
    raise RuntimeError("ffmpeg failed")

# ── Step 3: Collect scene metadata ───────────────────────────────────────────
print("PROGRESS|75|Building scenes...")
total_duration = get_duration(VIDEO_PATH)
boundaries = [0.0] + cut_points + [total_duration]
scenes = []
for idx in range(len(boundaries) - 1):
    clip_path  = os.path.join(OUTPUT_DIR, f"{FILE_NAME}_{idx:04d}.mp4")
    thumb_path = os.path.join(OUTPUT_DIR, f"{FILE_NAME}_{idx:04d}.jpg")
    if os.path.exists(clip_path) and os.path.getsize(clip_path) > 0:
        scenes.append({"scene_index": idx, "clip_path": clip_path, "thumb_path": thumb_path})

total = len(scenes)
print(f"  {total} scenes collected")

# ── Step 4: Generate thumbnails + pair results ────────────────────────────────
print(f"PROGRESS|90|Generating thumbnails... 0/{total}")

position_ready = [False] * total
next_pair_pos  = [0]
lock = threading.Lock()

def make_thumbnail(clip_path: str, thumb_path: str) -> None:
    try:
        with av.open(clip_path) as container:
            stream = container.streams.video[0]
            stream.codec_context.skip_frame = "NONKEY"
            for frame in container.decode(stream):
                img = frame.to_image()
                h = max(1, int(THUMB_WIDTH * img.height / img.width))
                img = img.resize((THUMB_WIDTH, h), resample=PILImage.Resampling.BICUBIC)
                img.save(thumb_path, "JPEG", quality=THUMB_QUALITY)
                return
    except Exception as e:
        print(f"  Thumbnail failed for {clip_path}: {e}")

def try_advance_pairs() -> None:
    while next_pair_pos[0] < total - 1:
        pa = next_pair_pos[0]
        pb = pa + 1
        if not (position_ready[pa] and position_ready[pb]):
            break
        should_merge = check_pair_similar(scenes[pa]["thumb_path"], scenes[pb]["thumb_path"])
        print(f"PAIR_RESULT|{pa}|{pb}|{'True' if should_merge else 'False'}")
        next_pair_pos[0] += 1

def build_one(args):
    pos, scene = args
    make_thumbnail(scene["clip_path"], scene["thumb_path"])
    # print(f"THUMBNAIL_READY|{pos}")
    with lock:
        position_ready[pos] = True
        try_advance_pairs()

progress_step = max(1, total // 25)
with ThreadPoolExecutor(max_workers=MAX_THUMB_WORKERS) as executor:
    futures = {executor.submit(build_one, (i, s)): i for i, s in enumerate(scenes)}
    done = 0
    for future in as_completed(futures):
        done += 1
        try:
            future.result()
        except Exception as e:
            print(f"  Worker error: {e}")
        if done % progress_step == 0 or done == total:
            print(f"PROGRESS|90|Generating thumbnails... {done}/{total}")

with lock:
    try_advance_pairs()

print(f"\nDone. {total} clips → {OUTPUT_DIR}/")

PROGRESS|10|Extracting keyframes...
  144228 keyframes found
PROGRESS|50|Cutting 3004 scenes...
PROGRESS|75|Building scenes...
  3005 scenes collected
PROGRESS|90|Generating thumbnails... 0/3005
CSM_buggy_movie_0000.jpg ↔ CSM_buggy_movie_0001.jpg | sim=1.0000  dissim=0.0000  → MERGE
PAIR_RESULT|0|1|True
CSM_buggy_movie_0001.jpg ↔ CSM_buggy_movie_0002.jpg | sim=0.9457  dissim=0.0543  → MERGE
PAIR_RESULT|1|2|True
CSM_buggy_movie_0002.jpg ↔ CSM_buggy_movie_0003.jpg | sim=0.9634  dissim=0.0366  → MERGE
PAIR_RESULT|2|3|True
CSM_buggy_movie_0003.jpg ↔ CSM_buggy_movie_0004.jpg | sim=0.8764  dissim=0.1236  → CUT
PAIR_RESULT|3|4|False
CSM_buggy_movie_0004.jpg ↔ CSM_buggy_movie_0005.jpg | sim=0.5784  dissim=0.4216  → CUT
PAIR_RESULT|4|5|False
CSM_buggy_movie_0005.jpg ↔ CSM_buggy_movie_0006.jpg | sim=0.7039  dissim=0.2961  → CUT
PAIR_RESULT|5|6|False
CSM_buggy_movie_0006.jpg ↔ CSM_buggy_movie_0007.jpg | sim=0.3321  dissim=0.6679  → CUT
PAIR_RESULT|6|7|False
CSM_buggy_movie_0007.jpg ↔ CSM_buggy_mo